In [1]:
import numpy as np
from collections import Counter

In [2]:
%store -r raw_data

In [3]:
def create_vocab(words, min_count = 5):
    counts = Counter(words)

    vocab = [word for word, freq in counts.items() if freq >= min_count]

    word_to_id = {word: i for i, word in enumerate(vocab)}

    id_to_word = {i: word for word, i in word_to_id.items()}

    return word_to_id, id_to_word

In [4]:
def words_to_ids(words, word_to_id):
    return [word_to_id[w] for w in words if w in word_to_id]

In [5]:
word_to_id, id_to_word = create_vocab(raw_data, min_count = 5)
data_as_ids = words_to_ids(raw_data, word_to_id)

<p>Subsampling helps with reducing frequent words from the vocabulary without loosing precision.</p>

In [6]:
def subsample_data(data_as_ids, threshold = 1e-5):
    counts = np.bincount(data_as_ids)
    total_words = len(data_as_ids)
    
    freqs = counts / total_words + 1e-9
    
    keep_probs = (np.sqrt(freqs / threshold) + 1) * (threshold / freqs)
    keep_probs = np.clip(keep_probs, 0.0, 1.0)
    
    random_draws = np.random.rand(len(data_as_ids))
    word_probs = keep_probs[data_as_ids]
    
    keep_mask = random_draws < word_probs
    subsampled_data = np.array(data_as_ids)[keep_mask]
    
    return subsampled_data

<p>Larger embed size leads to more nuanced connections, but for a showcase EMBED_SIZE = 100 will suffice.<br><br>WINDOW_SIZE means we are looking at 5 words to the left of the center and 5 words to the right of the center.<br><br>Negatives are random words taken in order to train negative examples and NEGATIVE_SIZE = 5 is right for this data and purpose.<br><br>Larger batch size speeds up the process, while higher learning rates make the model learn faster, but with the caveat of approaching infinite values, thus making the model useless.</p>

In [7]:
VOCAB_SIZE = len(word_to_id)
EMBED_SIZE = 100
WINDOW_SIZE = 5
NEGATIVE_SIZE = 5
BATCH_SIZE = 2048
LEARNING_RATE = 0.025
EPOCHS = 3

In [8]:
%run ../models/skipgram_model.ipynb
model = SkipgramModel(VOCAB_SIZE, EMBED_SIZE, LEARNING_RATE)

In [9]:
def get_all_pairs(data_as_ids, window_size):
    data = np.array(data_as_ids)
    all_centers = []
    all_contexts = []
    
    for offset in range(-window_size, window_size + 1):
        if offset == 0: continue
        context = np.roll(data, -offset)
    
        if offset > 0:
            all_centers.append(data[:-offset])
            all_contexts.append(context[:-offset])
        else:
            all_centers.append(data[-offset:])
            all_contexts.append(context[-offset:])
            
    return np.concatenate(all_centers), np.concatenate(all_contexts)

In [10]:
clean_data_ids = subsample_data(data_as_ids, threshold = 1e-5)

In [11]:
full_centers, full_contexts = get_all_pairs(clean_data_ids, WINDOW_SIZE)

<p>Before running epochs, words are mixed up so that the model doesn't learn from word ordering.</p>

In [12]:
state = np.random.get_state()
np.random.shuffle(full_centers)
np.random.set_state(state)
np.random.shuffle(full_contexts)

<p>These 3 epochs finished in less than an hour on the subsampled data.</p>

In [13]:
train_limit = len(full_centers)
print(f"Model will train on {train_limit} instances.")

for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0

    
    for i in range(0, train_limit, BATCH_SIZE):
        c_batch = full_centers[i : i + BATCH_SIZE]
        p_batch = full_contexts[i : i + BATCH_SIZE]
        
        loss = model.train_step(c_batch, p_batch)
        
        steps += 1
        total_loss += loss
        
    print(f"Epoch {epoch+1} finished, avg. loss: {total_loss/steps}")

Model will train on 55676580 instances.
Epoch 1 finished, avg. loss: 1.811631441116333
Epoch 2 finished, avg. loss: 1.474876046180725
Epoch 3 finished, avg. loss: 1.4133884906768799


In [14]:
def get_vector(word, word_to_id, W):
    if word in word_to_id:
        return W[word_to_id[word]]
    return None

In [15]:
def most_similar(word, word_to_id, id_to_word, W, top_n = 5):
    v = get_vector(word, word_to_id, W)
    if v is None:
        return "Word not present"
    
    norm_W = W / np.linalg.norm(W, axis=1, keepdims=True)
    norm_v = v / np.linalg.norm(v)

    similarities = norm_W @ norm_v
    
    top_indices = np.argsort(similarities)[::-1][1:top_n+1]
    
    return [(id_to_word[idx], similarities[idx]) for idx in top_indices]

<p>"tea" works pretty well.</p>

In [16]:
print(most_similar("tea", word_to_id, id_to_word, model.W1))

[('boba', np.float32(0.7338791)), ('tapioca', np.float32(0.7211075)), ('teas', np.float32(0.6776884)), ('caramel', np.float32(0.66654253)), ('liqueur', np.float32(0.6625547))]


In [19]:
print(most_similar("queen", word_to_id, id_to_word, model.W1))

[('regnant', np.float32(0.7163271)), ('elizabeth', np.float32(0.706819)), ('monarch', np.float32(0.6853342)), ('princess', np.float32(0.6671622)), ('crown', np.float32(0.66505265))]


<p>"apple" produces instances connected to the company Apple, from which we can conclude that there are many articles written about "Apple products", rather than "apple products".</p>

In [20]:
print(most_similar("apple", word_to_id, id_to_word, model.W1))

[('iigs', np.float32(0.7723396)), ('macintosh', np.float32(0.7538733)), ('macintoshes', np.float32(0.751178)), ('prodos', np.float32(0.7510974)), ('appleworks', np.float32(0.72899354))]


<p>Similarly to "apple", the model learned about the connections that the color orange has, rather than the fruit itself.</p>

In [25]:
print(most_similar("orange", word_to_id, id_to_word, model.W1, 10))

[('white', np.float32(0.58243537)), ('magenta', np.float32(0.5716656)), ('green', np.float32(0.5576751)), ('conch', np.float32(0.5427982)), ('colour', np.float32(0.5419706)), ('yellow', np.float32(0.5394194)), ('chestnut', np.float32(0.5348531)), ('brightly', np.float32(0.533818)), ('riebeeck', np.float32(0.5292037)), ('angostura', np.float32(0.524896))]


In [38]:
print(most_similar("serbia", word_to_id, id_to_word, model.W1))

[('montenegro', np.float32(0.89958435)), ('yugoslavia', np.float32(0.80104625)), ('yugoslav', np.float32(0.80035347)), ('belgrade', np.float32(0.77514297)), ('vojvodina', np.float32(0.76820207))]
